In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

# Data cleaning

בדיקת כפילויות, חוסרים, רווחים

In [3]:
import pandas as pd

# Load CSV
file_path = "Coffee Shop Sales.csv"
df = pd.read_csv(file_path)

print("=== Basic info ===")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print("Columns:", list(df.columns))

# Normalize text cells: trim spaces and convert empty strings to NA
text_cols = df.select_dtypes(include="object").columns
df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())
df[text_cols] = df[text_cols].replace("", pd.NA)

print("\n=== Missing values (per column) ===")
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct.round(2)})
print(missing_report)

print("\n=== Duplicate rows ===")
full_row_duplicates = df.duplicated().sum()
print(f"Full duplicate rows: {full_row_duplicates:,}")

# Check duplicates in likely ID columns (if they exist)
id_like_cols = [c for c in df.columns if any(k in c.lower() for k in ["id", "invoice", "order", "transaction"])]
if id_like_cols:
    print("\n=== Duplicate values in ID-like columns ===")
    for col in id_like_cols:
        dup_count = df.duplicated(subset=[col]).sum()
        print(f"{col}: {dup_count:,} duplicate values")
else:
    print("\nNo ID-like columns found for key-duplicate checks.")

print("\n=== Sample rows with missing values ===")
print(df[df.isna().any(axis=1)].head(10))

print("\n=== Sample duplicate rows (full-row) ===")
print(df[df.duplicated(keep=False)].head(10))

=== Basic info ===
Rows: 149,116
Columns: 11
Columns: ['transaction_id', 'transaction_date', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']

=== Missing values (per column) ===


C:\Users\natal\AppData\Local\Temp\ipykernel_58268\1451175152.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


                  missing_count  missing_pct
transaction_id                0          0.0
transaction_date              0          0.0
transaction_time              0          0.0
transaction_qty               0          0.0
store_id                      0          0.0
store_location                0          0.0
product_id                    0          0.0
unit_price                    0          0.0
product_category              0          0.0
product_type                  0          0.0
product_detail                0          0.0

=== Duplicate rows ===
Full duplicate rows: 0

=== Duplicate values in ID-like columns ===
transaction_id: 0 duplicate values
transaction_date: 148,935 duplicate values
transaction_time: 123,354 duplicate values
transaction_qty: 149,110 duplicate values
store_id: 149,113 duplicate values
product_id: 149,036 duplicate values

=== Sample rows with missing values ===
Empty DataFrame
Columns: [transaction_id, transaction_date, transaction_time, transaction_qt

# Gaps

בדיקת מספרים רצים בטרנזקציות


In [5]:
# Check gaps in Transaction ID running numbers
candidate_cols = [
    c for c in df.columns
    if ("transaction" in c.lower() and "id" in c.lower())
]

if not candidate_cols:
    print("Transaction ID column was not found.")
else:
    tx_col = candidate_cols[0]
    print(f"Using column: {tx_col}")

    # Convert to string, trim edges, and detect internal spaces between digits
    tx_raw = df[tx_col].astype(str).str.strip()
    has_internal_spaces = tx_raw.str.contains(r"\d\s+\d", regex=True, na=False)

    internal_space_count = int(has_internal_spaces.sum())
    print(f"Rows with internal spaces in Transaction ID: {internal_space_count}")
    if internal_space_count > 0:
        print("Sample values with internal spaces:")
        print(df.loc[has_internal_spaces, tx_col].head(10))

    # Remove spaces and convert IDs to numeric for sequence-gap check
    tx_numeric = pd.to_numeric(
        tx_raw.str.replace(r"\s+", "", regex=True),
        errors="coerce"
    )

    valid_ids = tx_numeric.dropna().astype("Int64").sort_values().drop_duplicates()

    if len(valid_ids) < 2:
        print("Not enough valid numeric Transaction IDs to check sequence gaps.")
    else:
        ids_sorted = pd.Series(valid_ids.astype("int64").to_list())
        prev_ids = ids_sorted.shift(1)
        gap_mask = (ids_sorted - prev_ids) > 1

        if gap_mask.any():
            gap_report = pd.DataFrame({
                "previous_id": prev_ids[gap_mask].astype(int).values,
                "next_id": ids_sorted[gap_mask].astype(int).values
            })
            gap_report["missing_from"] = gap_report["previous_id"] + 1
            gap_report["missing_to"] = gap_report["next_id"] - 1
            gap_report["missing_count"] = gap_report["next_id"] - gap_report["previous_id"] - 1

            total_missing_ids = int(gap_report["missing_count"].sum())
            print(f"Total missing IDs in sequence: {total_missing_ids}")
            print(f"Number of gap segments: {len(gap_report)}")
            print("Gap summary (first 10):")
            print(gap_report[["missing_from", "missing_to", "missing_count"]])
        else:
            print("Total missing IDs in sequence: 0")
            print("Number of gap segments: 0")
            print("No gaps found in Transaction ID running numbers.")

Using column: transaction_id
Rows with internal spaces in Transaction ID: 0
Total missing IDs in sequence: 340
Number of gap segments: 12
Gap summary (first 10):
    missing_from  missing_to  missing_count
0           3252        3280             29
1          20651       20677             27
2          37660       37709             50
3          53755       53756              2
4          54396       54397              2
5          59765       59823             59
6          79622       79625              4
7          86623       86702             80
8         112014      112017              4
9         113040      113043              4
10        120744      120818             75
11        148358      148361              4


In [6]:
# Check gaps in Product ID running numbers
candidate_cols = [
    c for c in df.columns
    if ("product" in c.lower() and "id" in c.lower())
]

if not candidate_cols:
    print("Product ID column was not found.")
else:
    prod_col = candidate_cols[0]
    print(f"Using column: {prod_col}")

    # Convert to string, trim edges, and detect internal spaces between digits
    prod_raw = df[prod_col].astype(str).str.strip()
    has_internal_spaces = prod_raw.str.contains(r"\d\s+\d", regex=True, na=False)

    internal_space_count = int(has_internal_spaces.sum())
    print(f"Rows with internal spaces in Product ID: {internal_space_count}")
    if internal_space_count > 0:
        print("Sample values with internal spaces:")
        print(df.loc[has_internal_spaces, prod_col].head(10))

    # Remove spaces and convert IDs to numeric for sequence-gap check
    prod_numeric = pd.to_numeric(
        prod_raw.str.replace(r"\s+", "", regex=True),
        errors="coerce"
    )

    valid_ids = prod_numeric.dropna().astype("Int64").sort_values().drop_duplicates()

    if len(valid_ids) < 2:
        print("Not enough valid numeric Product IDs to check sequence gaps.")
    else:
        ids_sorted = pd.Series(valid_ids.astype("int64").to_list())
        prev_ids = ids_sorted.shift(1)
        gap_mask = (ids_sorted - prev_ids) > 1

        if gap_mask.any():
            gap_report = pd.DataFrame({
                "previous_id": prev_ids[gap_mask].astype(int).values,
                "next_id": ids_sorted[gap_mask].astype(int).values
            })
            gap_report["missing_from"] = gap_report["previous_id"] + 1
            gap_report["missing_to"] = gap_report["next_id"] - 1
            gap_report["missing_count"] = gap_report["next_id"] - gap_report["previous_id"] - 1

            total_missing_ids = int(gap_report["missing_count"].sum())
            print(f"Total missing IDs in sequence: {total_missing_ids}")
            print(f"Number of gap segments: {len(gap_report)}")
            print("Gap summary (first 10):")
            print(gap_report[["missing_from", "missing_to", "missing_count"]])
        else:
            print("Total missing IDs in sequence: 0")
            print("Number of gap segments: 0")
            print("No gaps found in Product ID running numbers.")

Using column: product_id
Rows with internal spaces in Product ID: 0
Total missing IDs in sequence: 7
Number of gap segments: 4
Gap summary (first 10):
   missing_from  missing_to  missing_count
0            62          62              1
1            66          68              3
2            80          80              1
3            85          86              2
